# S6E9 | Three CPU Baselines and a Paired AUC Comparison

**Question:** what is the measured trade-off between a prior-only classifier,
logistic regression and a small histogram-gradient-boosted model under the
**same** data split and compute constraints?

This is a **120,000-row development benchmark**, not a full-data result or
leaderboard submission. The stratified split has 96,000 training and 24,000
validation rows. Only official training data are read, and ID is excluded.
Imputation, scaling and one-hot category vocabularies are learned inside each
training pipeline. There is no test-aware encoding, source-data augmentation,
feature selection or early stopping on the scored holdout.

The predeclared primary contrast is **HGB minus logistic ROC-AUC**. A paired,
label-stratified bootstrap describes evaluation-row uncertainty for these
fitted models on this holdout; it does not include training/split variability.
No parameter tuning or production/submission decision is performed.

In [ ]:
OUTPUT_NAME = "ev_c_cpu_baselines"

import os, json, time, hashlib, platform, resource, sys
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

START = time.perf_counter()
override = os.environ.get("EV_DATA_DIR")
roots = [Path(override)] if override else [
    Path("/kaggle/input/competitions/playground-series-s6e9"),
    Path("/kaggle/input/playground-series-s6e9")]
paths = {p.resolve() for p in roots if (p/"train.csv").is_file()}
if len(paths) != 1:
    raise RuntimeError("Attach one official S6E9 competition input, or set EV_DATA_DIR locally")
TRAIN_PATH = next(iter(paths)) / "train.csv"
OUTPUT = Path(OUTPUT_NAME)
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
def file_hash(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()
manifest = {"observed_at_utc": datetime.now(timezone.utc).isoformat(),
    "train_sha256": file_hash(TRAIN_PATH), "python": platform.python_version(),
    "packages": {p: version(p) for p in ("pandas", "numpy", "scikit-learn", "matplotlib")}}
print(json.dumps(manifest, indent=2))

## 1. Explicit, inexpensive reference models

- Prior: a constant probability learned from training labels.
- Logistic: C=1, lbfgs, max_iter=500, tol=1e-4; numeric scaling is train-only.
- HGB: 120 iterations, learning_rate=0.08, 31 leaves, min_samples_leaf=40,
  l2_regularization=1, early_stopping=False, random_state=17.

These settings were fixed before execution. All pipelines use dense one-hot
categoricals, suitable for these low-cardinality inputs. BLAS/OpenMP threads
are limited to two during fitting and prediction. Fitted pipelines remain
available in memory for inspection, but no weights or row predictions are exported.

In [ ]:
"""Original train-only experimental helpers for EV-B and EV-C."""
import hashlib
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

TARGET, ID = "Will_Buy_EV", "id"
SAMPLE_SEED = 20260907
SAMPLE_SIZE = 120000
SEEDS = [17, 43, 91]


def load_sample(path, n=SAMPLE_SIZE, seed=SAMPLE_SEED):
    data = pd.read_csv(path)
    if TARGET not in data or ID not in data or not data[ID].is_unique:
        raise ValueError("Expected unique IDs and a training target")
    if data[TARGET].isna().any() or set(data[TARGET].unique()) != {"Yes", "No"}:
        raise ValueError("Expected Yes/No labels")
    if n < 10 or n > len(data):
        raise ValueError("Sample size must be between 10 and dataset size")
    indices = np.arange(len(data))
    if n < len(data):
        indices, _ = train_test_split(indices, train_size=n, random_state=seed, stratify=data[TARGET])
    sample = data.iloc[indices].sort_values(ID).reset_index(drop=True)
    return sample, len(data)


def split_indices(frame, mode, seed=17, fraction=.2):
    idx = np.arange(len(frame))
    if mode in {"stratified", "shuffled"}:
        tr, va = train_test_split(idx, test_size=fraction, random_state=seed,
                                 stratify=frame[TARGET] if mode == "stratified" else None)
    elif mode in {"id_tail", "id_head"}:
        order = np.argsort(frame[ID].to_numpy(), kind="stable")
        count = int(np.ceil(len(frame) * fraction))
        if mode == "id_tail":
            tr, va = order[:-count], order[-count:]
        else:
            tr, va = order[count:], order[:count]
    else:
        raise ValueError("Unknown validation mode")
    assert len(tr) and len(va) and not np.intersect1d(tr, va).size
    assert len(np.union1d(tr, va)) == len(frame)
    return tr, va


def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan


def pipeline_for(frame, kind):
    numeric = frame.select_dtypes(include="number").columns.tolist()
    categories = [c for c in frame if c not in numeric]
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if kind == "linear":
        numeric_steps.append(("scale", StandardScaler()))
    prepare = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categories),
    ])
    if kind == "prior":
        model = DummyClassifier(strategy="prior")
    elif kind == "linear":
        model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=500, tol=1e-4)
    elif kind == "tree":
        model = HistGradientBoostingClassifier(max_iter=120, learning_rate=.08,
            max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.,
            early_stopping=False, random_state=17)
    else:
        raise ValueError("Unknown model kind")
    return Pipeline([("prepare", prepare), ("model", model)])


def fit_score(sample, tr, va, kind):
    features = [c for c in sample if c not in (TARGET, ID)]
    xtr, xva = sample.iloc[tr][features], sample.iloc[va][features]
    ytr = sample.iloc[tr][TARGET].map({"No": 0, "Yes": 1}).to_numpy()
    yva = sample.iloc[va][TARGET].map({"No": 0, "Yes": 1}).to_numpy()
    if len(np.unique(ytr)) != 2 or len(np.unique(yva)) != 2:
        raise ValueError("Overall training and validation splits must contain both classes")
    model = pipeline_for(xtr, kind)
    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        start = time.perf_counter()
        model.fit(xtr, ytr)
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        predictions = model.predict_proba(xva)[:, list(model.classes_).index(1)]
        predict_seconds = time.perf_counter() - start
    return model, predictions, yva, {
        "train_rows": len(tr), "validation_rows": len(va),
        "train_positive_rate": float(ytr.mean()), "validation_positive_rate": float(yva.mean()),
        "roc_auc": safe_auc(yva, predictions),
        "average_precision": float(average_precision_score(yva, predictions)),
        "log_loss": float(log_loss(yva, predictions, labels=[0, 1])),
        "brier_score": float(brier_score_loss(yva, predictions)),
        "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
        "convergence_warnings": sum(issubclass(w.category, ConvergenceWarning) for w in caught),
        "other_warnings": sum(not issubclass(w.category, ConvergenceWarning) for w in caught),
    }


def split_fingerprint(sample, tr, va):
    def digest(idx):
        ids = np.sort(sample.iloc[idx][ID].to_numpy()).astype("<i8")
        return hashlib.sha256(ids.tobytes()).hexdigest()
    return {"training_id_sha256": digest(tr), "validation_id_sha256": digest(va)}


def paired_auc_bootstrap(y, candidate, reference, rounds=500, seed=2026):
    y = np.asarray(y)
    positive, negative = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if not len(positive) or not len(negative) or rounds < 20:
        raise ValueError("Both labels and at least 20 bootstrap rounds are required")
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(rounds):
        idx = np.concatenate([rng.choice(positive, len(positive), replace=True),
                              rng.choice(negative, len(negative), replace=True)])
        differences.append(roc_auc_score(y[idx], candidate[idx]) - roc_auc_score(y[idx], reference[idx]))
    return np.asarray(differences)

In [ ]:
manifest["packages"]["scipy"] = version("scipy")
print("Numerical environment:", manifest["packages"])

## 2. Freeze the sample and holdout

Sampling seed 20260907 and split seed 17 match the first stratified design in
EV-B. The split fingerprints permit comparison without exporting row IDs.
Do not keep modifying this benchmark until its holdout becomes a training set.

In [ ]:
sample, full_rows = load_sample(TRAIN_PATH)
manifest.update({"full_training_rows": full_rows, "sample_rows": len(sample),
    "sampling_seed": SAMPLE_SEED,
    "sample_id_sha256": hashlib.sha256(sample[ID].to_numpy().astype("<i8").tobytes()).hexdigest()})
print(json.dumps({k:v for k,v in manifest.items() if k!='packages'}, indent=2))
print("Sample label counts:", sample[TARGET].value_counts().to_dict())
print("Test file and sample_submission.csv are not loaded.")

In [ ]:
tr, va = split_indices(sample, "stratified", seed=17)
assert len(tr)==96000 and len(va)==24000 and not np.intersect1d(tr,va).size
fingerprint = split_fingerprint(sample, tr, va)
print(json.dumps(fingerprint, indent=2))
yy = np.array([0,1]*10)
pp = np.linspace(.1,.9,20)
assert np.all(paired_auc_bootstrap(yy, pp, pp, rounds=20) == 0)
print("Split and paired-bootstrap known-answer controls: PASS")

## 3. Fit and evaluate all three models

ROC-AUC and average precision are higher-is-better; log loss and Brier are
lower-is-better. Average precision is not trapezoidal PR-AUC. Fit and prediction
times are measured for this hardware and sample, not a general speed ranking.

In [ ]:
fitted, predictions, rows = {}, {}, []
for kind in ("prior", "linear", "tree"):
    fitted[kind], predictions[kind], yva, result = fit_score(sample, tr, va, kind)
    result["model"] = kind
    rows.append(result)
    print(f"{kind}: AUC={result['roc_auc']:.6f}, AP={result['average_precision']:.6f}, fit={result['fit_seconds']:.2f}s")
scores = pd.DataFrame(rows).set_index("model")
assert scores.convergence_warnings.sum()==0, "Inspect convergence before comparison"
assert abs(scores.loc['prior','roc_auc']-.5)<1e-12
display(scores.round(6))
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1,2,figsize=(11,4.2),layout="constrained")
colors = {"prior":"#52616B", "linear":"#16817A", "tree":"#D1603D"}
for kind, probs in predictions.items():
    fpr,tpr,_ = roc_curve(yva,probs)
    axes[0].plot(fpr,tpr,color=colors[kind],label=f"{kind}: {scores.loc[kind,'roc_auc']:.4f}")
axes[0].set(title="Fixed development holdout",xlabel="False positive rate",ylabel="True positive rate",xlim=(0,1),ylim=(0,1))
axes[0].legend()
pos=np.arange(len(scores))
axes[1].bar(pos-.18,scores.fit_seconds,width=.36,label="fit including preprocessing",color="#16817A")
axes[1].bar(pos+.18,scores.predict_seconds,width=.36,label="predict 24,000 rows",color="#D1603D")
axes[1].set_xticks(pos,scores.index)
axes[1].set(title="Measured CPU time",ylabel="Seconds")
axes[1].legend(fontsize=8)
fig.savefig(OUTPUT / "01_models_and_cost.png",bbox_inches="tight")
plt.show()

## 4. Paired AUC uncertainty, with its scope kept narrow

Use the **same resampled validation rows for both models**. Each of 500
replicates samples with replacement within each label, preserving the observed
class counts. The interval is the 2.5th and 97.5th percentiles of the paired
differences. It is conditional on this sample, split, class balance and fitted
pair; it is not an interval for a future Public/Private LB gain. No p-value or
claim of guaranteed superiority is produced. The seed is fixed at 2026.

In [ ]:
differences = paired_auc_bootstrap(yva,predictions['tree'],predictions['linear'],rounds=500,seed=2026)
delta=float(scores.loc['tree','roc_auc']-scores.loc['linear','roc_auc'])
lower,upper=np.quantile(differences,[.025,.975])
contrast={"comparison":"tree minus linear", "holdout_auc_difference":delta,
    "paired_bootstrap_rounds":500, "bootstrap_seed":2026,
    "percentile_95_lower":float(lower), "percentile_95_upper":float(upper),
    "scope":"Conditional on this fitted pair, sample, split and class counts; not training or LB uncertainty"}
display(pd.Series(contrast,name="value").to_frame())
fig,ax=plt.subplots(figsize=(9,3.8),layout="constrained")
ax.hist(differences,bins=30,color="#16817A",edgecolor="white")
ax.axvline(delta,color="#D1603D",label="Observed holdout difference")
ax.axvline(lower,color="#52616B",linestyle="--",label="Conditional percentile bounds")
ax.axvline(upper,color="#52616B",linestyle="--")
ax.set(title="Paired bootstrap: HGB minus logistic",xlabel="ROC-AUC difference (zoomed)",ylabel="Replicates")
ax.legend(fontsize=9)
fig.savefig(OUTPUT / "02_paired_auc_uncertainty.png",bbox_inches="tight")
plt.show()
display(Markdown(f"Observed difference: **{delta:+.6f}**; conditional percentile interval "
    f"**[{lower:+.6f}, {upper:+.6f}]**. This does not authorize model promotion or a competition submission."))

## 5. What this benchmark does and does not establish

It provides a reproducible CPU starting point with explicit preprocessing,
a meaningful constant baseline and a matched holdout comparison. It does not
measure full-data performance, near-duplicate robustness or test generalization.
A different sample, stronger feature engineering or full-data training can
change the ordering. Before claiming a robust improvement, use a separately
reserved evaluation and multiple justified training/split repetitions.

Only two model families are tested, so this is not a comprehensive model search.
The reference hyperparameters are not offered as optimized values. Report
unfavorable results rather than deleting a losing model from the comparison.

**Environment check (2026-09-07):** during preparation, the same sample, split
and code produced HGB AUC 0.940998 in the initial local stack (NumPy 2.5.3,
pandas 3.0.5, SciPy 1.18.1, scikit-learn 1.9.0), versus 0.941179 on Kaggle
(NumPy 2.0.2, pandas 2.3.3, SciPy 1.16.3, scikit-learn 1.6.1). Logistic AUC
agreed. A local check with Kaggle-matched numerical library versions reproduced
Kaggle's model metrics and paired interval. This multi-package check does not
isolate the cause to one library. The live tables above always recompute scores
in the displayed runtime; do not combine results from different environments
as though they came from one fitted model. Solver deprecation warnings are
counted separately from convergence warnings in the score table.

In [ ]:
summary={"candidate":"EV-C", "manifest":manifest, "partition_fingerprint":fingerprint,
    "scores":scores.reset_index().to_dict(orient='records'), "primary_contrast":contrast,
    "test_data_read":False,"submission_created":False,"leaderboard_score":None,
    "total_experiment_seconds":round(time.perf_counter()-START,2),
    "process_peak_rss_mib":round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/(1024**2 if sys.platform=='darwin' else 1024),2)}
scores.to_csv(OUTPUT / "model_scores.csv")
(OUTPUT / "summary.json").write_text(json.dumps(summary,indent=2,allow_nan=False))
print(json.dumps({k:v for k,v in summary.items() if k not in ('manifest','scores')},indent=2))
print("No raw rows, row-level predictions, model weights or submission file exported.")

## Sources, reuse and boundaries

- [Official S6E9 competition](https://www.kaggle.com/competitions/playground-series-s6e9),
  Yao Yan, Walter Reade and Elizabeth Park, Kaggle, 2026;
  [data](https://www.kaggle.com/competitions/playground-series-s6e9/data) and
  [rules](https://www.kaggle.com/competitions/playground-series-s6e9/rules).
- Earlier [EV-A data-contract audit](https://www.kaggle.com/code/muelsyse111/s6e9-data-audit-and-id-aware-drift)
  found no exact full-feature duplicates. This is not proof that near-duplicates
  or latent groups do not exist.
- Related reading: Georgy Mamarin,
  [S6E9 starter: how to tell a real gain from noise](https://www.kaggle.com/code/georgymamarin/s6e9-starter-how-to-tell-a-real-gain-from-noise).
  Validation noise already has public coverage; no first-discovery claim is made here.
- Related baseline: evgendvorkin,
  [S6E9 Single XGB CV](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94583).
  Its title and reported scores are not our measurements or comparable benchmarks.
  Neither reference's code is copied or executed in this work.
- Models, preprocessing, splitting and AUC/AP/Brier/log-loss metrics use scikit-learn.

Original research code was prepared with AI assistance. Each notebook is independently
runnable from the official training input. Only aggregate statistics, split fingerprints
and plots are exported, not raw records, row predictions, model weights or submission files.
Synthetic competition relationships do not establish causal effects or real-world forecasts.